# 第 20 章：数值方法

本章的每一种方法，都用在同一份合约上：六个月的看跌期权，$S_0 = K = 50$，
$r = 5\%$，$\sigma = 30\%$。把合约固定下来，方法之间才可比。

| 小节 | 方法 | 例 |
|---|---|---|
| 1 | 二叉树，以及它的振荡 | 表 20.2 |
| 2 | 美式看跌期权上的控制变量 | 20.1 |
| 3 | 现金股息的三种处理 | 20.2 |
| 4 | 蒙特卡洛的收敛 | 20.3 |
| 5 | 抵得上 5800 万条路径的控制变量 | 20.6 |
| 6 | 相对股东总回报业绩权益 | 20.7 |
| 7 | Longstaff–Schwartz 逐步演示 | 20.8、20.9 |
| 8 | 显式格式的不稳定 | 20.10 |
| 9 | 隐式格式与 Crank–Nicolson | 20.11 |
| 10 | 由树与由网格算出的希腊字母 | 20.12 |

确定性的数字（树、股息、有限差分）都精确重现。带模拟的那些各自带着自己的
随机种子，因此它们报出的是标准误差，比较时也按标准误差来比，而不是逐位比较。


In [1]:
import numpy as np
from math import log, sqrt, exp
from scipy.stats import norm

N = norm.cdf
S0, K, r, SIG, T = 50.0, 50.0, 0.05, 0.30, 0.5


def show(label, got, printed, tol=6e-5):
    """Print a computed value beside the one the chapter prints."""
    flag = "ok" if abs(got - printed) <= tol * max(1.0, abs(printed)) else "DIFFERS"
    print("%-42s %14.6f   book %12.6f   %s" % (label, got, printed, flag))


def bs_put(S, K, T, r, sig):
    d1 = (log(S / K) + (r + 0.5 * sig * sig) * T) / (sig * sqrt(T))
    return K * exp(-r * T) * N(-(d1 - sig * sqrt(T))) - S * N(-d1)


EXACT = bs_put(S0, K, T, r, SIG)
show("European put, Black-Scholes", EXACT, 3.582934)


European put, Black-Scholes                      3.582934   book     3.582934   ok


## 1. 二叉树，以及它的振荡（表 20.2）

误差并不平滑地下降。它随 $n$ 的奇偶而振荡，
因为是否有节点落在执行价上会随 $n$ 改变。
接下来两节的内容都由此而来。

In [2]:
def crr(n, american=False, S=S0, K=K, T=T, r=r, sig=SIG, div=None, t_div=None):
    """CRR 树。给定 (div, t_div) 时，树建在位移后的即期价上，
    而行权判断用真实股价，这就是第 20.5 节的托管股息处理。"""
    S_tree = S - div * exp(-r * t_div) if div else S
    dt = T / n
    u = exp(sig * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)
    j = np.arange(n + 1)
    v = np.maximum(K - S_tree * u ** (2 * j - n), 0.0)
    for i in range(n - 1, -1, -1):
        j = np.arange(i + 1)
        S_i = S_tree * u ** (2 * j - i)
        v = disc * (p * v[1:] + (1 - p) * v[:-1])
        if american:
            t = i * dt
            S_actual = S_i + (div * exp(-r * (t_div - t)) if div and t < t_div else 0.0)
            v = np.maximum(v, K - S_actual)
    return float(v[0])


print("%6s %12s %12s" % ("n", "tree", "error"))
for n, want in ((5, 3.789185), (10, 3.479418), (11, 3.675951), (20, 3.530789),
                (21, 3.631453), (50, 3.561990), (100, 3.572449), (101, 3.592980),
                (200, 3.577688), (201, 3.587979), (500, 3.580835), (1000, 3.581884)):
    v = crr(n)
    print("%6d %12.6f %+12.6f    book %.6f" % (n, v, v - EXACT, want))

     n         tree        error
     5     3.789185    +0.206251    book 3.789185
    10     3.479418    -0.103516    book 3.479418
    11     3.675951    +0.093017    book 3.675951
    20     3.530789    -0.052145    book 3.530789
    21     3.631453    +0.048519    book 3.631453
    50     3.561990    -0.020943    book 3.561990
   100     3.572449    -0.010485    book 3.572449
   101     3.592980    +0.010046    book 3.592980
   200     3.577688    -0.005246    book 3.577688
   201     3.587979    +0.005045    book 3.587979
   500     3.580835    -0.002099    book 3.580835
  1000     3.581884    -0.001050    book 3.581884


## 2. 美式看跌期权上的控制变量（例 20.1）

在*同一棵*树上给美式与欧式定价，再用欧式的已知误差去修正美式。
控制变量消去的是两个问题所**共有**的那部分误差，
所以 $n$ 为奇数时效果惊人，为偶数时作用不大。
基准值也出自同一做法，只是算得足够远：对 2,000 与 4,000 步各做修正，再在两者之间外推。

In [3]:
def corrected_tree(n):
    """美式树值加上树在欧式期权上的误差。"""
    return crr(n, american=True) + (EXACT - crr(n))


# 基准：先做修正，再在两个偶数步数之间做 Richardson 外推；
# 平值时两者的偏移量相同（第 20.3 节的生产做法）。
AMER = 2 * corrected_tree(4000) - corrected_tree(2000)
show("American put, benchmark", AMER, 3.69702)
show("early exercise premium", AMER - EXACT, 0.114, tol=5e-3)

print()
print("%5s %12s %12s %12s %12s" % ("n", "plain", "error", "corrected", "error"))
for n in (50, 51, 100, 101, 200, 201):
    plain = crr(n, american=True)
    corrected = corrected_tree(n)
    print("%5d %12.5f %+12.5f %12.5f %+12.5f" % (n, plain, plain - AMER, corrected, corrected - AMER))
print()
for n in (200, 201):
    print("n = %d：修正把误差缩小为原来的 1/%.1f"
          % (n, abs((crr(n, american=True) - AMER) / (corrected_tree(n) - AMER))))

American put, benchmark                          3.697021   book     3.697020   ok
early exercise premium                           0.114087   book     0.114000   ok

    n        plain        error    corrected        error
   50      3.68552     -0.01150      3.70646     +0.00944
   51      3.71691     +0.01989      3.69700     -0.00002
  100      3.69135     -0.00567      3.70183     +0.00481
  101      3.70717     +0.01015      3.69712     +0.00010
  200      3.69419     -0.00283      3.69944     +0.00242
  201      3.70213     +0.00511      3.69709     +0.00007

n = 200：修正把误差缩小为原来的 1/1.2
n = 201：修正把误差缩小为原来的 1/76.9


## 3. 现金股息的三种处理（例 20.2）

把树建在 $S^{*} = S_0 - D e^{-rt_d}$ 上，而行权判断对照*真实*股价。
忽略股息是灾难性的，而且一眼可见；
减去未贴现的股息则是一个小的、单方向的误差，会随利率与时间放大。

In [4]:
D, T_DIV = 2.00, 0.25
S_star = S0 - D * exp(-r * T_DIV)
show("displaced spot S*", S_star, 48.0248, tol=1e-5)

for n in (200, 2000, 20001):
    print("  n=%6d   %.4f" % (n, crr(n, american=True, div=D, t_div=T_DIV)))

correct = crr(20001, american=True, div=D, t_div=T_DIV)
euro = bs_put(S_star, K, T, r, SIG)
print()
show("American, displaced tree (correct)", correct, 4.6098, tol=2e-4)
show("European, Black-Scholes on S*", euro, 4.4689, tol=1e-4)
show("dividend ignored", AMER, 3.6970, tol=2e-4)
show("spot reduced by the full 2.00", crr(20001, american=True, S=48.0), 4.6385, tol=2e-4)
print()
print("忽略股息:          %+.1f%%" % (100 * (AMER / correct - 1)))
print("整额减去 2.00:     %+.1f%%" % (100 * (crr(20001, american=True, S=48.0) / correct - 1)))
show("early exercise premium with the dividend", correct - euro, 0.141, tol=2e-2)

displaced spot S*                               48.024844   book    48.024800   ok
  n=   200   4.6143
  n=  2000   4.6101


  n= 20001   4.6097



American, displaced tree (correct)               4.609743   book     4.609800   ok
European, Black-Scholes on S*                    4.468861   book     4.468900   ok
dividend ignored                                 3.697021   book     3.697000   ok


spot reduced by the full 2.00                    4.638377   book     4.638500   ok

忽略股息:          -19.8%


整额减去 2.00:     +0.6%
early exercise premium with the dividend         0.140882   book     0.141000   ok


### 同一笔股息的三种支付方式

*连续股息率*进入漂移。*离散的按比例股息*把 $t_d$ 上每个节点都乘以同一个
$1-\delta$，于是格点是被整体缩放而不是被扭曲，不需要任何位移。
只有*现金*股息会破坏重合性，因为减去一个常数会让各个节点移动的比例互不相同。

把金额固定在今天价格的 4%，就能看出各种选择值多少：
支付的形式让价值变动 0.3%，支付的时点让它变动 3% 以上。


In [5]:
# The same 4% dividend paid three ways. A proportional dividend's present
# value is exactly delta * S0, whatever the price does, because
# exp(-r t) S_t is a martingale; 2.00 of cash at t_d is worth less.
DELTA = 0.04
q_equiv = -log(1 - DELTA) / T


def crr_scaled(n, delta=0.0, q=0.0, american=True):
    """Every node from t_div onward carries the same factor 1 - delta, so
    the lattice is rescaled rather than distorted and still recombines.
    With q instead, the dividend is spread smoothly through the drift."""
    dt = T / n
    u = exp(SIG * sqrt(dt))
    d = 1.0 / u
    p = (exp((r - q) * dt) - d) / (u - d)
    disc = exp(-r * dt)

    def spot(i):
        s = S0 * u ** (2 * np.arange(i + 1) - i)
        return s * (1 - delta) if i * dt >= T_DIV else s

    v = np.maximum(K - spot(n), 0.0)
    for i in range(n - 1, -1, -1):
        v = disc * (p * v[1:] + (1 - p) * v[:-1])
        if american:
            v = np.maximum(v, K - spot(i))
    return v[0]


show("PV of the 2.00 cash dividend", D * exp(-r * T_DIV), 1.9752, tol=1e-4)
show("PV of the 4% proportional dividend", DELTA * S0, 2.0000, tol=1e-9)
show("equivalent continuous yield, %", 100 * q_equiv, 8.16, tol=1e-3)
print()

# Both forms leave the same terminal distribution, 0.96 S0 exp(...), so one
# closed form serves for the European value; only early exercise differs.
euro96 = bs_put(S0 * (1 - DELTA), K, T, r, SIG)
prem_lump = crr_scaled(2000, delta=DELTA) - crr_scaled(2000, delta=DELTA, american=False)
prem_yield = crr_scaled(2000, q=q_equiv) - crr_scaled(2000, q=q_equiv, american=False)

show("European, either form of the 4%", euro96, 4.4810, tol=1e-4)
show("American, 4% proportional at t_d", euro96 + prem_lump, 4.6223, tol=2e-4)
show("American, 4% as a continuous yield", euro96 + prem_yield, 4.4812, tol=2e-4)
show("early-exercise premium, lump at t_d", prem_lump, 0.1414, tol=5e-4)
show("early-exercise premium, continuous", prem_yield, 0.0002, tol=5e-4)
print()
print("形式: %+.1f%%  (按比例 vs. 2.00 现金)"
      % (100 * ((euro96 + prem_lump) / correct - 1)))
print("时点: %+.1f%%  (一次性支付 vs. 同样的 4%% 摊开)"
      % (100 * ((euro96 + prem_lump) / (euro96 + prem_yield) - 1)))


PV of the 2.00 cash dividend                     1.975156   book     1.975200   ok
PV of the 4% proportional dividend               2.000000   book     2.000000   ok
equivalent continuous yield, %                   8.164399   book     8.160000   ok

European, either form of the 4%                  4.480959   book     4.481000   ok
American, 4% proportional at t_d                 4.622425   book     4.622300   ok
American, 4% as a continuous yield               4.481172   book     4.481200   ok
early-exercise premium, lump at t_d              0.141466   book     0.141400   ok
early-exercise premium, continuous               0.000213   book     0.000200   ok

形式: +0.3%  (按比例 vs. 2.00 现金)
时点: +3.2%  (一次性支付 vs. 同样的 4% 摊开)


## 4. 蒙特卡洛的收敛（例 20.3）

路径数每增加十倍，标准误差下降 $\sqrt{10}$ 倍，与预期完全一致，
而实际误差在它的范围内来回跳。一百万条路径只把一份 \$3.58 期权的标准误差压到半美分，
而一棵 200 步的树已经在一美分之内。对一份普通欧式期权，蒙特卡洛是错误的工具。

In [6]:
def mc_put(n_paths, seed=1):
    rng = np.random.default_rng(seed)
    z = rng.standard_normal(n_paths)
    S_T = S0 * np.exp((r - 0.5 * SIG ** 2) * T + SIG * sqrt(T) * z)
    pay = np.maximum(K - S_T, 0.0) * exp(-r * T)
    return pay.mean(), pay.std() / sqrt(n_paths)


print("%12s %12s %16s %14s" % ("paths", "estimate", "standard error", "actual error"))
for n in (1_000, 10_000, 100_000, 1_000_000):
    m, se = mc_put(n)
    print("%12s %12.5f %16.5f %+14.5f" % ("{:,}".format(n), m, se, m - EXACT))
print()
print("一棵 200 步的树只用一小部分时间，误差就只有 %+.5f" % (crr(200) - EXACT))

       paths     estimate   standard error   actual error
       1,000      3.75377          0.16256       +0.17083
      10,000      3.62993          0.05058       +0.04700
     100,000      3.59128          0.01590       +0.00834


   1,000,000      3.57558          0.00503       -0.00735



一棵 200 步的树只用一小部分时间，误差就只有 -0.00525


## 5. 抵得上 5800 万条路径的控制变量（例 20.6）

算术平均亚式期权没有公式；*几何*平均那一版有，
因为对数正态的几何平均仍是对数正态。
两者的收益几乎完全相关，于是几何版抵消了算术版的大部分误差。

例 20.1 中树的精度，也来自同一模式。

In [7]:
S0A, KA, rA, sigA, TA, NFIX = 50.0, 50.0, 0.05, 0.30, 1.0, 12
t_fix = np.arange(1, NFIX + 1) * TA / NFIX


def geometric_asian_closed_form():
    mu = log(S0A) + (rA - 0.5 * sigA ** 2) * t_fix.mean()
    var = sigA ** 2 / NFIX ** 2 * sum((2 * i - 1) * t_fix[NFIX - i] for i in range(1, NFIX + 1))
    sg = sqrt(var)
    d1 = (mu - log(KA) + var) / sg
    return exp(-rA * TA) * (exp(mu + 0.5 * var) * N(d1) - KA * N(d1 - sg))


GEO = geometric_asian_closed_form()
show("geometric Asian, closed form", GEO, 4.012352)


def asian_mc(n_paths=100_000, seed=5):
    rng = np.random.default_rng(seed)
    dt = TA / NFIX
    z = rng.standard_normal((n_paths, NFIX))
    S = S0A * np.exp(np.cumsum((rA - 0.5 * sigA ** 2) * dt + sigA * sqrt(dt) * z, axis=1))
    arith = np.maximum(S.mean(axis=1) - KA, 0.0) * exp(-rA * TA)
    geo = np.maximum(np.exp(np.log(S).mean(axis=1)) - KA, 0.0) * exp(-rA * TA)
    beta = np.cov(arith, geo)[0, 1] / geo.var()
    cv = arith - beta * (geo - GEO)
    rho = np.corrcoef(arith, geo)[0, 1]
    return arith, cv, rho, beta


arith, cv, rho, beta = asian_mc()
n = len(arith)
print()
print("%-28s %12s %14s" % ("", "estimate", "standard error"))
print("%-28s %12.5f %14.5f" % ("plain Monte Carlo", arith.mean(), arith.std() / sqrt(n)))
print("%-28s %12.5f %14.5f" % ("geometric control variate", cv.mean(), cv.std() / sqrt(n)))
print()
print("相关系数 rho = %.4f, beta = %.4f" % (rho, beta))
factor = arith.var() / cv.var()
print("方差下降为原来的 1/%.0f" % factor)
print("要达到同样的标准误差，普通蒙特卡洛需要 %.0f 百万条路径"
      % (n * factor / 1e6))

geometric Asian, closed form                     4.012352   book     4.012352   ok



                                 estimate standard error
plain Monte Carlo                 4.25150        0.02031
geometric control variate         4.23600        0.00084

相关系数 rho = 0.9991, beta = 1.0484
方差下降为原来的 1/588
要达到同样的标准误差，普通蒙特卡洛需要 59 百万条路径


## 6. 相对股东总回报业绩权益（例 20.7）

取得的单位数取决于公司相对指数的回报，而结算以股票进行。
朴素的算法 $E[u]\,E[S_T]$ 把价值低估了 30%（正确值比它高 44%），
因为单位数恰恰是在股价高的那些路径上取得的。差额完全来自协方差。

In [8]:
def tsr_award(n_paths=400_000, seed=9):
    S0T, sig_c, sig_i, rho_ci, rT, TT = 50.0, 0.35, 0.20, 0.60, 0.05, 3.0
    rng = np.random.default_rng(seed)
    z1 = rng.standard_normal(n_paths)
    z2 = rho_ci * z1 + sqrt(1 - rho_ci ** 2) * rng.standard_normal(n_paths)
    R_c = np.exp((rT - 0.5 * sig_c ** 2) * TT + sig_c * sqrt(TT) * z1)
    R_i = np.exp((rT - 0.5 * sig_i ** 2) * TT + sig_i * sqrt(TT) * z2)
    rel = R_c - R_i                                   # 相对股东总回报
    units = np.clip(1.0 + rel / 0.30, 0.0, 2.0)       # -30% 处为 0%，0 处为 100%，+30% 处封顶 200%
    S_T = S0T * R_c
    return units, S_T, exp(-rT * TT)


units, S_T, disc = tsr_award()
show("expected units earned", units.mean(), 0.8361, tol=2e-2)
show("expected share price", S_T.mean(), 58.1795, tol=2e-2)
show("naive product, disc * E[u] * E[S_T]", disc * units.mean() * S_T.mean(), 41.8704, tol=3e-2)
show("correct value, disc * E[u * S_T]", disc * (units * S_T).mean(), 60.2468, tol=3e-2)
print()
print("正确值比朴素算法高 %.0f%%"
      % (100 * ((units * S_T).mean() / (units.mean() * S_T.mean()) - 1)))
print("而且这份权益比一股股票（50.00）更值钱")

expected units earned                            0.837308   book     0.836100   ok
expected share price                            58.193814   book    58.179500   ok
naive product, disc * E[u] * E[S_T]             41.938985   book    41.870400   ok
correct value, disc * E[u * S_T]                60.275830   book    60.246800   ok

正确值比朴素算法高 44%
而且这份权益比一股股票（50.00）更值钱


## 7. 一条路径，反向计算（例 20.8）

取本章的看涨期权（$S_0 = K = 100$，$r = 10\%$，$\sigma = 20\%$，$T = 1$），只抽一次
$\varepsilon = 0.5$。正向一遍保存全部中间值；反向一遍从 $\bar V = 1$ 出发，每一步乘一个
局部导数，每个输入在它进入计算的那一步读出自己的敏感度。最后几行改用扰动重估逐个输入
计算：要多定价四次，才得到反向一遍已经给出的结果。

In [9]:
def call_aad(S0c, K_c, r_c, sig_c, T_c, eps):
    """看涨期权的一条路径：先正向一遍并保存，再反向一遍。"""
    # 正向一遍：保存每一个中间值
    x = (r_c - 0.5 * sig_c ** 2) * T_c + sig_c * sqrt(T_c) * eps
    S_Tc = S0c * exp(x)
    disc_c = exp(-r_c * T_c)
    V = disc_c * max(S_Tc - K_c, 0.0)
    # 反向一遍：从 V_bar = 1 出发，每一步乘一个局部导数
    V_bar = 1.0
    S_T_bar = V_bar * disc_c * (1.0 if S_Tc > K_c else 0.0)
    x_bar = S_T_bar * S_Tc
    grads = {
        "S0": S_T_bar * S_Tc / S0c,
        "sigma": x_bar * (sqrt(T_c) * eps - sig_c * T_c),
        "r": x_bar * T_c - T_c * V,           # r 出现两次：在漂移项中，也在贴现因子中
        "K": -S_T_bar,
    }
    return x, S_Tc, V, S_T_bar, x_bar, grads


EPS1 = 0.5
BASE = dict(S0c=100.0, K_c=100.0, r_c=0.10, sig_c=0.20, T_c=1.0, eps=EPS1)
x1, ST1, V1, ST_bar, x_bar, g = call_aad(**BASE)
show("x", x1, 0.18, tol=5e-4)
show("S_T", ST1, 119.72, tol=5e-4)
show("V", V1, 17.84, tol=5e-4)
show("adjoint of S_T", ST_bar, 0.9048, tol=5e-4)
show("adjoint of x", x_bar, 108.33, tol=5e-4)
show("dV/dS0", g["S0"], 1.083, tol=5e-4)
show("dV/dsigma", g["sigma"], 32.50, tol=5e-4)
show("dV/dr", g["r"], 90.48, tol=5e-4)
show("dV/dK", g["K"], -0.905, tol=5e-4)

# 用扰动重估得到同样四个数：每个输入多定价一次。
h = 1e-6
print()
print("%-8s %14s %14s" % ("input", "adjoint", "bump"))
for name, key in (("S0", "S0c"), ("sigma", "sig_c"), ("r", "r_c"), ("K", "K_c")):
    up = dict(BASE)
    up[key] += h
    bump = (call_aad(**up)[2] - V1) / h
    print("%-8s %14.6f %14.6f" % (name, g[name], bump))
print("一次反向传递给出全部四个；扰动重估多定价了四次")

x                                                0.180000   book     0.180000   ok
S_T                                            119.721736   book   119.720000   ok
V                                               17.844965   book    17.840000   ok
adjoint of S_T                                   0.904837   book     0.904800   ok
adjoint of x                                   108.328707   book   108.330000   ok
dV/dS0                                           1.083287   book     1.083000   ok
dV/dsigma                                       32.498612   book    32.500000   ok
dV/dr                                           90.483742   book    90.480000   ok
dV/dK                                           -0.904837   book    -0.905000   ok

input           adjoint           bump
S0             1.083287       1.083287
sigma         32.498612      32.498563
r             90.483742      90.483697
K             -0.904837      -0.904837
一次反向传递给出全部四个；扰动重估多定价了四次


## 8. 美式蒙特卡洛逐步演示（例 20.9）

模拟路径处在中间某个日期时，并不知道自己的未来，
而提前行权的决策恰恰需要这个。Longstaff–Schwartz 用**跨路径回归**
来估计继续持有的价值：许多路径会经过邻近的状态，
因此这个横截面里包含了这一条件期望。

第 20.11 节的四个步骤，从到期日往回做：

1. 只取当前**实值**的路径，因为其余路径上的决策是显然的；
2. 把已实现的、贴现后的继续持有现金流对 $1, S, S^2$ 做回归；
3. 把拟合值当作继续持有的价值，凡立即行权收益超过它的路径即行权；
4. 把现金流往回带一步，重复。

接下来三个单元先慢慢做一遍，再看那次回归究竟决定了什么，最后正式跑一遍。

In [10]:
def lsm_paths(n_paths, n_dates, seed=13):
    """行权网格上的对偶几何布朗运动路径。第 i 列对应日期 (i+1)*dt。"""
    rng = np.random.default_rng(seed)
    dt = T / n_dates
    z = rng.standard_normal((n_paths, n_dates))
    z = np.concatenate([z, -z])
    return S0 * np.exp(np.cumsum((r - 0.5 * SIG ** 2) * dt + SIG * sqrt(dt) * z, axis=1)), dt


def longstaff_schwartz(S, dt, report_at=None):
    """The four steps of Section 20.11. Returns the value, and optionally the
    diagnostics from one exercise date so the regression can be inspected."""
    n_dates = S.shape[1]
    cash = np.maximum(K - S[:, -1], 0.0)              # 到期时的价值
    boundary, snapshot = np.full(n_dates, np.nan), None
    for i in range(n_dates - 2, -1, -1):
        cash *= exp(-r * dt)                          # 第 4 步：往回带一步
        exercise = np.maximum(K - S[:, i], 0.0)
        itm = exercise > 0                            # 第 1 步：只取实值路径
        if itm.sum() > 10:
            x = S[itm, i]
            design = np.vstack([np.ones_like(x), x, x * x]).T
            coef, *_ = np.linalg.lstsq(design, cash[itm], rcond=None)   # step 2
            continuation = design @ coef
            take = exercise[itm] > continuation       # 第 3 步
            idx = np.where(itm)[0][take]
            cash[idx] = exercise[idx]
            if take.any():
                boundary[i] = x[take].max()           # 行权仍占优的最高 S
            if report_at is not None and i == report_at:
                snapshot = dict(i=i, S=x, cash=cash[itm].copy(), coef=coef,
                                continuation=continuation, exercise=exercise[itm],
                                take=take)
    cash *= exp(-r * dt)
    # 对偶路径成对出现（第 i 行与第 i + half 行），所以标准误差
    # 每对只计一次，而不是把两条路径当作独立样本。
    half = len(cash) // 2
    pairs = 0.5 * (cash[:half] + cash[half:])
    return cash.mean(), pairs.std() / sqrt(half), boundary, snapshot


S_paths, dt = lsm_paths(20_000, 25)
value, se, boundary, snap = longstaff_schwartz(S_paths, dt, report_at=12)
print("价值 %.5f   标准误差 %.5f   二叉树基准 %.5f" % (value, se, AMER))

价值 3.66510   标准误差 0.01240   二叉树基准 3.69702


### 那次回归在某一个日期上决定了什么

25 个日期中的第 13 个，大约在中途。拟合出来的二次式就是估计的继续持有价值；
直线 $K - S$ 是立即行权收益。在收益高于拟合曲线的地方，方法选择行权，
两者相交之处就是该日期上的行权边界。

注意参与这个决策的路径有多少：只有实值的那些被拿去回归，
这正是 Longstaff 与 Schwartz 的核心手法。
把其余路径也放进去，只会把拟合的自由度花在一个答案早已知道的区域上。

留意表中实值程度最深的那一行。在那里拟合说*继续持有*，而显然应当行权：
二次式在那么远的地方几乎没有数据，外推得很差。
这在这里几乎不花什么代价，因为到得了那里的路径很少；
但实务上正因为如此，才要增加基函数，或者直接把收益本身作为回归元。

In [11]:
a, b, c = snap["coef"]
print("date %d of %d, t = %.3f" % (snap["i"] + 1, S_paths.shape[1], (snap["i"] + 1) * dt))
print("paths in the money: %s of %s" % ("{:,}".format(len(snap["S"])),
                                        "{:,}".format(len(S_paths))))
print("fitted continuation  E[V | S] = %.4f %+.4f S %+.6f S^2" % (a, b, c))
print("exercised on %s of them" % "{:,}".format(int(snap["take"].sum())))
print()
print("%8s %14s %16s %12s" % ("S", "exercise now", "fitted continue", "decision"))
order = np.argsort(snap["S"])
for j in order[::max(1, len(order) // 12)]:
    print("%8.2f %14.4f %16.4f %12s"
          % (snap["S"][j], snap["exercise"][j], snap["continuation"][j],
             "EXERCISE" if snap["take"][j] else "hold"))

# 两条曲线在 c S^2 + (b+1) S + (a-K) = 0 处相交。二次拟合与一条直线相交两次；
# 边界取较大的那个根，也就是行权仍胜过继续持有的最高 S。
#
roots = np.roots([c, b + 1.0, a - K])
roots = sorted(x.real for x in roots if abs(x.imag) < 1e-9 and 0 < x.real <= K)
print()
print("两条曲线的交点: %s" % ", ".join("%.2f" % x for x in roots))
print("边界取较大的那个根，S = %.2f：低于它行权，高于它继续持有" % roots[-1])
print("路径的结果与之一致：实际行权的最高 S 为 %.2f" % boundary[snap["i"]])

date 13 of 25, t = 0.260
paths in the money: 19,859 of 40,000
fitted continuation  E[V | S] = 75.8745 -2.3801 S +0.018231 S^2
exercised on 4,164 of them

       S   exercise now  fitted continue     decision
   27.17        22.8338          24.6722         hold
   38.43        11.5720          11.3360     EXERCISE
   40.51         9.4876           9.3750     EXERCISE
   41.95         8.0456           8.1109         hold
   43.14         6.8631           7.1310         hold
   44.21         5.7862           6.2829         hold
   45.15         4.8474           5.5780         hold
   46.01         3.9875           4.9606         hold
   46.83         3.1733           4.4009         hold
   47.66         2.3412           3.8538         hold
   48.44         1.5565           3.3610         hold
   49.22         0.7772           2.8939         hold
   50.00         0.0034           2.4519         hold

两条曲线的交点: 34.20, 41.49
边界取较大的那个根，S = 41.49：低于它行权，高于它继续持有
路径的结果与之一致：实际行权的最高 S 为 41.49


### 它勾出来的边界，以及它带着的偏差

有两件事值得一看。回归还原出一条随到期临近而向执行价上升的**行权边界**：
树逐个节点定位的也是这条曲线，而这里只用路径就找到了它。

而偏差有*两个*，方向相反。在**同一批**路径上既拟合规则又估值，
会让规则用到这些路径本不该拥有的信息，把答案往上推；
而不完美的行权规则把它往下推。交易台用新的路径估值来把两者分开。
这个差距要量出来而不是假定：在三个基函数、四万条路径之下，
这里的前视偏差小到可以忽略，下面两个值之差只是标准误差的一小部分。

In [12]:
print("回归勾出来的行权边界")
print("%10s %12s" % ("t", "boundary S"))
for i in range(0, S_paths.shape[1] - 1, 4):
    if not np.isnan(boundary[i]):
        print("%10.3f %12.2f" % ((i + 1) * dt, boundary[i]))

print()
print("样本内与样本外")
S_fit, _ = lsm_paths(20_000, 25, seed=13)
S_new, _ = lsm_paths(20_000, 25, seed=99)
in_sample, se_in, _, _ = longstaff_schwartz(S_fit, dt)
out_sample, se_out, _, _ = longstaff_schwartz(S_new, dt)
print("  在同一批路径上拟合并估值  %.5f  (%+.5f)" % (in_sample, in_sample - AMER))
print("  在独立路径上估值          %.5f  (%+.5f)" % (out_sample, out_sample - AMER))
print("  二叉树基准                %.5f" % AMER)

回归勾出来的行权边界
         t   boundary S
     0.180        40.63
     0.260        41.49
     0.340        41.97
     0.420        42.98

样本内与样本外


  在同一批路径上拟合并估值  3.66510  (-0.03192)
  在独立路径上估值          3.66585  (-0.03117)
  二叉树基准                3.69702


### 例 20.10 的那张表

行权日为 25 个或 50 个时，该方法给出的是百慕大期权的价格，
所以每个估计都与一棵只能在这些日期行权的树作比较。
任何非最优的行权规则赚得都更少，因此这个方法向下有偏；
但在这些路径数下，偏差与一个标准误差同量级：单独一行证明不了它，
各行向两个方向的散布都有一个标准误差甚至更多。
增加路径只会缩小标准误差，不会去掉偏差；更好的基函数能减小偏差，
更多的行权日则使基准更接近美式期权。
路径数按书中的口径以对偶路径计，标准误差中每对只计一次。

对一份一维美式期权，这是严格意义上的错误选择：
一棵 200 步的树只用一小部分时间就胜过它。
它的价值在于：树建不起来时它仍然能用。

In [13]:
def bermudan(n_dates, per_date):
    """只能在 n_dates 个行权日行权的看跌期权，在 CRR 树上定价。"""
    n = n_dates * per_date
    dt = T / n
    u = exp(SIG * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)
    v = np.maximum(K - S0 * u ** (2 * np.arange(n + 1) - n), 0.0)
    for i in range(n - 1, -1, -1):
        v = disc * (p * v[1:] + (1 - p) * v[:-1])
        if i > 0 and i % per_date == 0:
            v = np.maximum(v, K - S0 * u ** (2 * np.arange(i + 1) - i))
    return float(v[0])


# 每个行权日之间取 80 与 160 个树步，做 Richardson 外推
BERM = {nd: 2 * bermudan(nd, 160) - bermudan(nd, 80) for nd in (25, 50)}
show("Bermudan put, 25 exercise dates", BERM[25], 3.6907)
show("Bermudan put, 50 exercise dates", BERM[50], 3.6938)
print()
print("%10s %8s %12s %12s %12s %12s"
      % ("paths", "dates", "estimate", "std error", "Bermudan", "error"))
for n_paths, n_dates in ((20_000, 25), (100_000, 25), (100_000, 50), (400_000, 50)):
    S_i, dt_i = lsm_paths(n_paths // 2, n_dates)      # n_paths 按对偶路径对计
    m, s, _, _ = longstaff_schwartz(S_i, dt_i)
    print("%10s %8d %12.5f %12.4f %12.4f %+12.4f"
          % ("{:,}".format(n_paths), n_dates, m, s, BERM[n_dates], m - BERM[n_dates]))
print()
print("一棵 200 步的树只用一小部分时间，误差就只有 %+.5f"
      % (crr(200, american=True) - AMER))

Bermudan put, 25 exercise dates                  3.690741   book     3.690700   ok
Bermudan put, 50 exercise dates                  3.693841   book     3.693800   ok

     paths    dates     estimate    std error     Bermudan        error


    20,000       25      3.69681       0.0175       3.6907      +0.0061


   100,000       25      3.66563       0.0079       3.6907      -0.0251


   100,000       50      3.68495       0.0077       3.6938      -0.0089


   400,000       50      3.68705       0.0038       3.6938      -0.0068

一棵 200 步的树只用一小部分时间，误差就只有 -0.00283


## 9. 不稳定的表现（例 20.11）

只加密空间网格而不动时间网格，会把整个计算毁掉。
条件大致是 $\Delta t \lesssim 1/(\sigma^2 M^2)$，
因此**空间分辨率翻倍，就需要四倍的时间步**。

有一点可以宽心：不稳定的显式格式失败得很明显。
发散的具体量级取决于边界处理；可复现的是它错得离谱，而不是错一点点。

In [14]:
def fd_put(M, n_t, theta, Smax=200.0, american=True):
    """theta = 0 显式，1 隐式，0.5 Crank-Nicolson。"""
    dS, dt = Smax / M, T / n_t
    S = np.linspace(0.0, Smax, M + 1)
    v = np.maximum(K - S, 0.0)
    j = np.arange(1, M)
    a = 0.5 * dt * (SIG ** 2 * j ** 2 - r * j)
    b = -dt * (SIG ** 2 * j ** 2 + r)
    c = 0.5 * dt * (SIG ** 2 * j ** 2 + r * j)
    if theta == 0.0:
        for _ in range(n_t):
            vn = v.copy()
            v[1:M] = a * vn[0:M - 1] + (1 + b) * vn[1:M] + c * vn[2:M + 1]
            v[0], v[M] = K, 0.0
            if american:
                v = np.maximum(v, K - S)
        return float(np.interp(S0, S, v))
    A = np.diag(1 - theta * b)
    B = np.diag(1 + (1 - theta) * b)
    for i in range(M - 1):
        if i > 0:
            A[i, i - 1], B[i, i - 1] = -theta * a[i], (1 - theta) * a[i]
        if i < M - 2:
            A[i, i + 1], B[i, i + 1] = -theta * c[i], (1 - theta) * c[i]
    for _ in range(n_t):
        rhs = B @ v[1:M]
        rhs[0] += a[0] * K
        v[1:M] = np.linalg.solve(A, rhs)
        v[0], v[M] = K, 0.0
        if american:
            v = np.maximum(v, K - S)
    return float(np.interp(S0, S, v))


print("%12s %12s %20s" % ("space steps", "time steps", "value"))
for M, n_t in ((100, 1000), (200, 1000), (200, 4000)):
    val = fd_put(M, n_t, 0.0)
    print("%12d %12d %20s" % (M, n_t, ("%.4f" % val) if abs(val) < 1e6 else "%.0e" % val))
print()
print("中间那一行不是错了一点点，而是错得离谱")

 space steps   time steps                value
         100         1000               3.6766
         200         1000                3e+58
         200         4000               3.6918

中间那一行不是错了一点点，而是错得离谱


## 10. 隐式格式与 Crank–Nicolson（例 20.12）

隐式格式的误差每加密一次大约减半，是一阶。
Crank–Nicolson 的误差依次缩小约 3.5 倍、3 倍：对称的时间平均在问题的光滑部分上是二阶的，
而每次求解之后施加的行权条件是一阶的，随着网格加密显露出来。
两者都从下方收敛，因为粗网格无法精确表示行权边界。

In [15]:
print("%8s %8s %14s %18s %14s" % ("space", "time", "implicit", "Crank-Nicolson", "CN error"))
for M, n_t, wi, wc in ((100, 50, 3.65776, 3.67363), (200, 100, 3.68196, 3.69033),
                       (400, 200, 3.69056, 3.69494)):
    imp, cn = fd_put(M, n_t, 1.0), fd_put(M, n_t, 0.5)
    print("%8d %8d %14.5f %18.5f %+14.5f    book %.5f / %.5f"
          % (M, n_t, imp, cn, cn - AMER, wi, wc))

   space     time       implicit     Crank-Nicolson       CN error
     100       50        3.65776            3.67363       -0.02339    book 3.65776 / 3.67363


     200      100        3.68196            3.69033       -0.00669    book 3.68196 / 3.69033


     400      200        3.69056            3.69494       -0.00208    book 3.69056 / 3.69494


## 11. 由树与由网格算出的希腊字母（例 20.13）

delta、gamma 与 theta 不额外花钱：两种方法都已经算出了作差所需的那些相邻取值。两个计算过程毫无共同之处的实现应当吻合，而这是检验它们最便宜的办法。vega 与 rho 是例外，这个单元格会显示它们的代价。


In [16]:
# Delta, gamma and theta cost nothing extra: both methods have already
# computed the neighboring values they are differences of. On a tree, the
# nodes at steps one and two; on a grid, the neighbors of S0 in the t = 0
# column and the column beside it.

def tree_greeks(n):
    """Value, delta, gamma and theta from the first two steps of a CRR tree."""
    dt = T / n
    u = exp(SIG * sqrt(dt))
    d = 1.0 / u
    p = (exp(r * dt) - d) / (u - d)
    disc = exp(-r * dt)
    v = np.maximum(K - S0 * u ** (2 * np.arange(n + 1) - n), 0.0)
    keep = {}
    for i in range(n - 1, -1, -1):
        v = disc * (p * v[1:i + 2] + (1.0 - p) * v[:i + 1])
        v = np.maximum(v, K - S0 * u ** (2 * np.arange(i + 1) - i))
        if i <= 2:
            keep[i] = v.copy()
    suu, sdd = S0 * u * u, S0 * d * d
    delta = (keep[1][1] - keep[1][0]) / (S0 * u - S0 * d)
    gamma = ((keep[2][2] - keep[2][1]) / (suu - S0)
             - (keep[2][1] - keep[2][0]) / (S0 - sdd)) / (0.5 * (suu - sdd))
    theta = (keep[2][1] - keep[0][0]) / (2.0 * dt)
    return float(keep[0][0]), float(delta), float(gamma), float(theta)


def fd_greeks(M, n_t, theta=0.5, Smax=200.0, sig=SIG, rate=r):
    """fd_put, keeping the last two columns so the Greeks can be read off."""
    dS, dt = Smax / M, T / n_t
    S = np.linspace(0.0, Smax, M + 1)
    payoff = np.maximum(K - S, 0.0)
    v = payoff.copy()
    j = np.arange(1, M)
    a = 0.5 * dt * (sig ** 2 * j ** 2 - rate * j)
    b = -dt * (sig ** 2 * j ** 2 + rate)
    c = 0.5 * dt * (sig ** 2 * j ** 2 + rate * j)
    A = np.diag(1 - theta * b)
    B = np.diag(1 + (1 - theta) * b)
    for i in range(M - 1):
        if i > 0:
            A[i, i - 1], B[i, i - 1] = -theta * a[i], (1 - theta) * a[i]
        if i < M - 2:
            A[i, i + 1], B[i, i + 1] = -theta * c[i], (1 - theta) * c[i]
    prev = v.copy()
    for _ in range(n_t):
        rhs = B @ v[1:M]
        rhs[0] += a[0] * K
        prev = v.copy()
        v = v.copy()
        v[1:M] = np.linalg.solve(A, rhs)
        v[0], v[M] = K, 0.0
        v = np.maximum(v, payoff)
    j0 = int(round(S0 / dS))
    delta = (v[j0 + 1] - v[j0 - 1]) / (2.0 * dS)
    gamma = (v[j0 + 1] - 2.0 * v[j0] + v[j0 - 1]) / dS ** 2
    return (float(v[j0]), float(delta), float(gamma),
            float((prev[j0] - v[j0]) / dt))


print("%-26s %10s %10s %10s %10s"
      % ("method", "value", "delta", "gamma", "theta"))
for label, got in (("tree, 6,000 steps", tree_greeks(6000)),
                   ("Crank-Nicolson, 800x400", fd_greeks(800, 400))):
    print("%-26s %10.4f %10.4f %10.5f %10.4f" % ((label,) + got))

d1 = (log(S0 / K) + (r + 0.5 * SIG ** 2) * T) / (SIG * sqrt(T))
euro_gamma = exp(-0.5 * d1 * d1) / sqrt(2 * np.pi) / (S0 * SIG * sqrt(T))
print("%-26s %10s %10.4f %10.5f"
      % ("European, closed form", "", N(d1) - 1.0, euro_gamma))

# Volatility and the rate are parameters, not state variables, so neither is
# a difference of grid values: each costs a second solve, here on the
# 800 x 400 grid the example tabulates.
base = fd_greeks(800, 400)[0]
print()
print("vega, per volatility point   %+.4f"
      % (fd_greeks(800, 400, sig=SIG + 0.01)[0] - base))
print("rho,  per basis point        %+.4f"
      % (fd_greeks(800, 400, rate=r + 0.0001)[0] - base))


method                          value      delta      gamma      theta


tree, 6,000 steps              3.6969    -0.4300    0.03953    -3.1873
Crank-Nicolson, 800x400        3.6963    -0.4299    0.03953    -3.1898
European, closed form                    -0.4114    0.03668


vega, per volatility point   +0.1373


rho,  per basis point        -0.0009
